In [ ]:
# # ================================
# # Qwen 2.5 VL
# # ================================
# model_name = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-32B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-72B-Instruct-bnb-4bit"

# # ================================
# # Qwen 2 VL
# # ================================
# model_name = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit"

# # ================================
# # Qwen 2.5 Omni (Multimodal)
# # ================================
# model_name = "unsloth/Qwen2.5-Omni-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-Omni-7B-Instruct-bnb-4bit"

# # ================================
# # Llama Vision
# # ================================
# model_name = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit"
# model_name = "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit"

# # ================================
# # Gemma Vision
# # ================================
# model_name = "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit"
# model_name = "unsloth/MedGemma-27B-Vision-Instruct-bnb-4bit"

# # ================================
# # Mistral Vision
# # ================================
# model_name = "unsloth/Pixtral-12B-2409-bnb-4bit"

# # ================================
# # LLaVA
# # ================================
# model_name = "unsloth/llava-1.5-7b-hf-bnb-4bit"
# model_name = "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit"


Example 1 – Simple Image + Instruction (Conversation Style)

{
  "image": "...",
  "conversations": [...]
}

Example 2 – VQA Format

{
  "image": "...",
  "question": "...",
  "answer": "..."
}

Example 3 – LLaVA Format (<image> token style)

{
  "conversations": [
    {"from": "human", "value": "<image>\n..."},
    {"from": "gpt", "value": "..."}
  ]
}


In [ ]:
"qwen2_vl_2b"
"qwen25_vl_3b"
"qwen25_omni_3b"
"medgemma_4b_vision"
"qwen25_vl_7b"
"qwen2_vl_7b"
"qwen25_omni_7b"
"llava15_7b"
"llava16_mistral_7b"

In [ ]:
# --- Install (Colab) ---
!pip -q install "transformers==4.57.1" --upgrade
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer


✔ Vision Layers → LoRA applied

✔ Language Layers → LoRA applied

✔ Attention → LoRA applied

✔ MLP → LoRA applied

Matlab:

You are NOT fully training them.

You are training LoRA adapters inside:

Vision encoder

LLM layers

Attention blocks

MLP blocks

| Strategy                            | Tumhara Code |
| ----------------------------------- | ------------ |
| Option 1 (Only projection)          | ❌ No         |
| Option 2 (Projection + LLM)         | ⚠️ Partially |
| Option 3 (Train everything full)    | ❌ No         |
| Option 4 (LoRA-based multimodal FT) | ✅ YES        |


Final Answer

👉 Tum LoRA-based multimodal fine-tuning kar rahe ho
👉 Vision + Language dono pe LoRA adapters train ho rahe hain
👉 Base weights freeze hain
👉 Full model train nahi ho raha

💡 Important Insight

Agar tum chahte:

Only projection fine-tune →

finetune_vision_layers=False

finetune_language_layers=False


Only LLM LoRA →

finetune_vision_layers=False

finetune_language_layers=True


Full heavy training →
LoRA hata ke full float16 training karni padti

https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K

https://huggingface.co/datasets/HuggingFaceM4/ChartQA

https://huggingface.co/datasets/nlphuji/flickr30k


In [ ]:
# ==========================================================
# Universal Unsloth Vision Fine-tuning (Model + Dataset Dynamic)
# Supports: Qwen VL, LLaVA, Pixtral, MedGemma, Llama Vision
# Dataset selectable
# Trains on 150 row subset
# Runs for 2 Epochs
# ==========================================================

import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict

from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig


# ==========================================================
# 1️Model Registry
# ==========================================================
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}


# ==========================================================
# Dataset Registry (Vision-ready datasets)
# ==========================================================
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },
    "vqav2": {
        "name": "HuggingFaceM4/VQAv2",
        "split": "train",
        "image_key": "image",
        "text_key": "question",
        "instruction": "Answer the question about the image."
    },

    "iphone_custom": {
    "name": "sunny199/iphone5_vlm",  # change to your HF repo
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

In [ ]:
# ==========================================================
# Config
# ==========================================================
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "latex_ocr"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

In [ ]:
from datasets import load_dataset
dataset = load_dataset("HuggingFaceM4/ChartQA")
print(dataset)
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
print(train_ds[0])

In [ ]:
# ==========================================================
# Trainer Class
# ==========================================================
class VisionFineTuner:

    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

    # -----------------------------
    # Load Model
    # -----------------------------
    def load_model(self):
        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            self.model_name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth"
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(format_sample, remove_columns=raw.column_names)

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):

        FastVisionModel.for_training(self.model)

        args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(self.model, self.tokenizer),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=args,
        )

        return self

    # -----------------------------
    # Train
    # -----------------------------
    def train(self):
        print("Training for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()

    # -----------------------------
    # Save
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)
        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)
        print("Saved to:", self.cfg.save_dir)

    # -----------------------------
    # Quick Inference
    # -----------------------------
    def quick_infer(self, sample_index=0):

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": self.dataset_info["instruction"]}
            ]}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(self.tokenizer, skip_prompt=True)

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
        )

In [ ]:
# ==========================================================
# Run
# ==========================================================

cfg = VisionFTConfig(
  model_key="qwen2_vl_2b",   # Change model here
  dataset_key="latex_ocr",  # Change dataset here
)

trainer = (
  VisionFineTuner(cfg)
  .load_model()
  .prepare_data()
  .build_trainer()
)

trainer.train()
trainer.save()
trainer.quick_infer()